# 4. Process PyNNLF Output: AEDP Aggregation Levels

Creates summary tables after the aggregation-level PyNNLF experiments have been run. This version expects 3 samples per aggregation level, `ds25` through `ds36`.


## 1. Setup And Paths

In [1]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
RESULTS_DIR = PROJECT_DIR / "results" / "03_aedp_aggregation_level"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RECAP_PATH = PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv"
DATA_EXPLORATION_DIR = RESULTS_DIR / "01_data_exploration"
SAMPLE_DESIGN_PATH = DATA_EXPLORATION_DIR / "aedp_aggregation_sample_design.csv"
MODEL_ORDER = ["m1_naive_hp1", "m6_lr_hp1", "m17_xgb_hp1"]
FH8_MINUTES = 1440
SAMPLES_PER_LEVEL = 3
AGGREGATION_LEVELS = [1, 10, 100, 1000]
DATASET_IDS = [f"ds{i}" for i in range(25, 25 + SAMPLES_PER_LEVEL * len(AGGREGATION_LEVELS))]

Publication project: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1
Repository root: C:\Users\z5404477\Documents\PyNNLF


## 2. Load Recap And Sample Design

In [2]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(f"Missing recap at {RECAP_PATH}. Run notebook 3 first.")
if not SAMPLE_DESIGN_PATH.exists():
    raise FileNotFoundError(f"Missing sample design at {SAMPLE_DESIGN_PATH}. Run notebook 2.2 first.")

recap = pd.read_csv(RECAP_PATH)
sample_design = pd.read_csv(SAMPLE_DESIGN_PATH)
expected_dataset_ids = DATASET_IDS
actual_design_ids = sample_design["dataset_id"].astype(str).tolist()
if actual_design_ids != expected_dataset_ids:
    raise ValueError(
        "Sample design does not match the expected 3-sample dataset range. "
        f"Expected {expected_dataset_ids[0]} through {expected_dataset_ids[-1]}, "
        f"found {actual_design_ids[:3]} ... {actual_design_ids[-3:]}"
    )
level_counts = sample_design.groupby("aggregation_level_hh")["dataset_id"].nunique().to_dict()
if level_counts != {level: SAMPLES_PER_LEVEL for level in AGGREGATION_LEVELS}:
    raise ValueError(f"Expected {SAMPLES_PER_LEVEL} samples per aggregation level, found {level_counts}")

rows = recap.loc[
    recap["dataset_no"].astype(str).isin(expected_dataset_ids)
    & pd.to_numeric(recap["forecast_horizon_min"], errors="coerce").eq(FH8_MINUTES)
].copy()
# The sample-design audit also has a numeric dataset_no column for human scanning.
# Drop it before merging so the PyNNLF recap dataset_no remains the canonical ds25-ds36 ID.
sample_design_for_merge = sample_design.drop(columns=["dataset_no"], errors="ignore")
rows = rows.merge(sample_design_for_merge, left_on="dataset_no", right_on="dataset_id", how="left")
if rows["aggregation_level_hh"].isna().any():
    missing_design = sorted(rows.loc[rows["aggregation_level_hh"].isna(), "dataset_no"].astype(str).unique())
    raise ValueError(f"Some recap rows did not match the sample design: {missing_design}")
rows["model_name"] = pd.Categorical(rows["model_name"], categories=MODEL_ORDER, ordered=True)
expected = {(dataset_id, model) for dataset_id in expected_dataset_ids for model in MODEL_ORDER}
actual = set(zip(rows["dataset_no"].astype(str), rows["model_name"].astype(str)))
missing = sorted(expected - actual)
if missing:
    raise ValueError(f"AEDP aggregation results are incomplete. Missing {len(missing)} combinations: {missing[:10]}")
if rows.shape[0] != len(expected):
    raise ValueError(f"Expected {len(expected)} recap rows, found {rows.shape[0]}")

display(rows[["dataset_no", "aggregation_level_hh", "sample_no", "model_name", "test_nRMSE", "test_nRMSE_stddev"]].head())


,dataset_no,aggregation_level_hh,sample_no,model_name,test_nRMSE,test_nRMSE_stddev
0,ds25,1,1,m1_naive_hp1,22.889373,4.474711
1,ds25,1,1,m6_lr_hp1,19.377828,5.003024
2,ds25,1,1,m17_xgb_hp1,19.707035,5.636853
3,ds26,1,2,m1_naive_hp1,25.422936,1.399729
4,ds26,1,2,m6_lr_hp1,20.562193,1.431863


## 3. Build Summary Tables

In [3]:
rows["runtime_s"] = pd.to_numeric(rows["runtime_ms"], errors="coerce") / 1000.0
summary = rows.groupby(["aggregation_level_hh", "model_name"], observed=False).agg(
    mean_test_nRMSE=("test_nRMSE", "mean"),
    sample_std_test_nRMSE=("test_nRMSE", "std"),
    mean_test_nRMSE_stddev=("test_nRMSE_stddev", "mean"),
    mean_runtime_s=("runtime_s", "mean"),
    n_samples=("dataset_no", "nunique"),
).reset_index()
wide_mean = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_test_nRMSE").reindex(MODEL_ORDER)
wide_sample_std = summary.pivot(index="model_name", columns="aggregation_level_hh", values="sample_std_test_nRMSE").reindex(MODEL_ORDER)
wide_cv_stddev = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_test_nRMSE_stddev").reindex(MODEL_ORDER)
wide_runtime = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_runtime_s").reindex(MODEL_ORDER)
wide_completed = summary.pivot(index="model_name", columns="aggregation_level_hh", values="n_samples").reindex(MODEL_ORDER)

PAPER_MODEL_LABELS = {"m1_naive_hp1": "naive_hp1", "m6_lr_hp1": "lr_hp1", "m17_xgb_hp1": "xgb_hp1"}
paper_rows = []
for model in MODEL_ORDER:
    row = {"Model Name": PAPER_MODEL_LABELS.get(model, model)}
    for level in AGGREGATION_LEVELS:
        row[f"{level}hh Mean Test nRMSE (%)"] = wide_mean.loc[model, level]
        row[f"{level}hh Sample Std Test nRMSE (%)"] = wide_sample_std.loc[model, level]
        row[f"{level}hh Mean Test nRMSE Stddev (%)"] = wide_cv_stddev.loc[model, level]
        row[f"{level}hh Mean Training Time (s)"] = wide_runtime.loc[model, level]
        row[f"{level}hh Completed Samples"] = wide_completed.loc[model, level]
    paper_rows.append(row)
paper_table = pd.DataFrame(paper_rows)
for column in paper_table.columns.drop("Model Name"):
    if column.endswith("Completed Samples"):
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").astype("Int64")
    elif column.endswith("Training Time (s)"):
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").round(1)
    else:
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").round(2)

rows.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_recap.csv", index=False)
summary.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_summary_by_level_model.csv", index=False)
wide_mean.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_nrmse_by_level.csv")
wide_sample_std.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_sample_std_nrmse_by_level.csv")
wide_cv_stddev.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_cv_stddev_by_level.csv")
wide_runtime.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_runtime_seconds_by_level.csv")
paper_table.to_csv(RESULTS_DIR / "paper_table_aedp_aggregation_levels.csv", index=False)

display(summary.round(3))
display(wide_mean.round(3))
display(paper_table)

,aggregation_level_hh,model_name,mean_test_nRMSE,sample_std_test_nRMSE,mean_test_nRMSE_stddev,mean_runtime_s,n_samples
0,1,m1_naive_hp1,20.647,6.209,2.905,0.002,3
1,1,m6_lr_hp1,16.956,5.254,2.970,0.817,3
2,1,m17_xgb_hp1,16.392,5.201,3.154,30.234,3
3,10,m1_naive_hp1,18.941,4.009,2.530,0.001,3
4,10,m6_lr_hp1,16.058,3.469,1.999,0.703,3
5,10,m17_xgb_hp1,14.529,2.367,1.747,26.570,3
6,100,m1_naive_hp1,23.482,0.750,2.936,0.002,3
7,100,m6_lr_hp1,20.210,0.630,2.409,0.616,3
8,100,m17_xgb_hp1,17.285,0.545,1.950,24.117,3
9,1000,m1_naive_hp1,24.630,0.616,3.034,0.002,3


aggregation_level_hh,1,10,100,1000
model_name,,,,
m1_naive_hp1,20.647,18.941,23.482,24.630
m6_lr_hp1,16.956,16.058,20.210,21.234
m17_xgb_hp1,16.392,14.529,17.285,18.274


,Model Name,1hh Mean Test nRMSE (%),1hh Sample Std Test nRMSE (%),1hh Mean Test nRMSE Stddev (%),1hh Mean Training Time (s),1hh Completed Samples,10hh Mean Test nRMSE (%),10hh Sample Std Test nRMSE (%),10hh Mean Test nRMSE Stddev (%),10hh Mean Training Time (s),...,100hh Mean Test nRMSE (%),100hh Sample Std Test nRMSE (%),100hh Mean Test nRMSE Stddev (%),100hh Mean Training Time (s),100hh Completed Samples,1000hh Mean Test nRMSE (%),1000hh Sample Std Test nRMSE (%),1000hh Mean Test nRMSE Stddev (%),1000hh Mean Training Time (s),1000hh Completed Samples
0,naive_hp1,20.65,6.21,2.91,0.0,3,18.94,4.01,2.53,0.0,...,23.48,0.75,2.94,0.0,3,24.63,0.62,3.03,0.0,3
1,lr_hp1,16.96,5.25,2.97,0.8,3,16.06,3.47,2.00,0.7,...,20.21,0.63,2.41,0.6,3,21.23,0.53,2.46,0.9,3
2,xgb_hp1,16.39,5.20,3.15,30.2,3,14.53,2.37,1.75,26.6,...,17.28,0.55,1.95,24.1,3,18.27,0.46,2.03,28.5,3
